In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_PATH = PROJECT_ROOT / "src"

sys.path.append(str(SRC_PATH))

In [3]:
import pandas as pd
from churn.features import build_feature_set, encode_categorical_features

customers = pd.read_csv("../data/generated/customers_data.csv")
sales = pd.read_csv("../data/generated/sales_data.csv", parse_dates=["Date"])

reference_date = pd.Timestamp("2025-12-31")

features = build_feature_set(customers, sales, reference_date)
features_encoded = encode_categorical_features(features)

print(features_encoded.shape)

(1000, 18)


In [4]:
from sklearn.model_selection import train_test_split

X = features_encoded.drop(columns=["Customer_ID", "Churn"])
y = features_encoded["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Proportion churn train:", y_train.mean().round(3))
print("Proportion churn test:", y_test.mean().round(3))

Train: (800, 16) Test: (200, 16)
Proportion churn train: 0.408
Proportion churn test: 0.41


### Interprétation — chargement et split

- **(1000, 18)** : rechargement via `build_feature_set` + `encode_categorical_features`
  identique au résultat obtenu dans `eda_churn.ipynb` — confirme que le pipeline
  extrait dans `features.py` est bien reproductible, sans avoir eu à réexpliquer
  ou recoder la logique de leakage/encodage.
- **Train (800, 16) / Test (200, 16)** : split 80/20 comme prévu. 16 colonnes
  = 18 - Customer_ID (identifiant, exclu) - Churn (la cible, séparée dans y).
- **Proportion churn : 0.408 (train) vs 0.41 (test)** — quasi identiques,
  écart de 0.002. Confirme que `stratify=y` a bien fonctionné : le split
  respecte la proportion réelle du dataset (40.8% de churn), pas de biais
  introduit par le split lui-même.

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # apprend ET applique sur train
X_test_scaled = scaler.transform(X_test)         # applique seulement, n'apprend pas

model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

### Entraînement — Logistic Regression (baseline)

Modèle entraîné sur X_train_scaled/y_train (StandardScaler appris sur train
uniquement, appliqué sur test). Pas d'erreur, prêt pour évaluation.